## Evaluating function

In [7]:
import pandas as pd, torch
from tqdm import tqdm
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForMaskedLM, AutoConfig, logging
logging.set_verbosity_error()

# ----- NLL helpers -----------------------------------------------------------
def nll_autoregressive(model, tok, context, candidate):
    enc = tok(context + " " + candidate, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model(**enc, labels=enc["input_ids"])
    return float(out.loss * enc["input_ids"].size(1))      # Σ –log p

def nll_masked(model, tok, context, candidate):
    full   = tok(context + " " + candidate)
    ctx_sz = len(tok(context)["input_ids"]) - 1            # without BOS
    cand   = full["input_ids"][ctx_sz:]
    nll = 0.0
    for i, tgt in enumerate(cand):
        masked           = full["input_ids"].copy()
        masked[ctx_sz+i] = tok.mask_token_id
        enc = {
            "input_ids": torch.tensor([masked]).to(model.device),
            "attention_mask": torch.tensor([full["attention_mask"]]).to(model.device)
        }
        with torch.no_grad():
            logits = model(**enc).logits[0, ctx_sz+i]
        nll -= float(torch.log_softmax(logits, dim=-1)[tgt])
    return nll

# ----- main evaluation -------------------------------------------------------
def evaluate_cat(csv_path, model_name="gpt2", device=None, seed=42):

    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(seed)
    df = pd.read_csv(csv_path)

    # ---- build triplets S/A/M ----------------------------------------------
    triplets = []
    for _, row in df.iterrows():
        triplets.append({
            "context": row["context"],
            "S": row["stereotype"],
            "A": row["anti-stereotype"],
            "M": row["meaningless"]
        })

    # ---- model & tokenizer --------------------------------------------------
    cfg       = AutoConfig.from_pretrained(model_name)
    causal    = cfg.is_decoder or cfg.model_type in {"gpt2","llama","mistral"}
    model_cls = AutoModelForCausalLM if causal else AutoModelForMaskedLM
    scorer    = nll_autoregressive if causal else nll_masked
    model     = model_cls.from_pretrained(model_name).eval().to(device)
    tok       = AutoTokenizer.from_pretrained(model_name)

    # ---- scoring ------------------------------------------------------------
    results, wins_s, wins_a = [], 0, 0
    for t in tqdm(triplets, desc="scoring"):
        scores = {k: scorer(model, tok, t["context"], cand) for k, cand in {"S":t["S"],"A":t["A"],"M":t["M"]}.items()}
        best   = min(scores, key=scores.get)                # lower NLL = better
        wins_s += best=="S"; wins_a += best=="A"
        results.append({**t, "prediction":best, "scores":scores})

    # ---- metrics ------------------------------------------------------------
    lms       = (wins_s + wins_a) / len(results) * 100
    ss        = wins_s / (wins_s + wins_a) * 100 if (wins_s+wins_a) else 0
    icat      = lms * (min(ss,100-ss)/50)
    return lms, ss, icat



## Run the evaluation

In [13]:
data_file  = Path.cwd().parent / "data" / "dataset_6.csv"
model_file = Path.cwd().parent / "models" / "bert_inlp_gender"

lms1, ss1, icat1 = evaluate_cat(str(data_file), "bert-base-uncased")
lms2, ss2, icat2 = evaluate_cat(str(data_file), str(model_file))

print(f"BEFORE ☠️\n  LMS  = {lms1:5.2f}   SS = {ss1:5.2f}   ICAT = {icat1:5.2f} \n\n"
      f"AFTER  ❤️\n  LMS  = {lms2:5.2f}   SS = {ss2:5.2f}   ICAT = {icat2:5.2f}")

BEFORE ☠️
  LMS  = 79.50   SS = 72.96   ICAT = 43.00 

AFTER  ❤️
  LMS  = 52.00   SS = 50.96   ICAT = 51.00
